# Colab setup

Run once per Colab session (T4 GPU, standard RAM). Takes ~5 minutes, mostly installing.

**Before the first run, put two things in Google Drive** (folder `MyDrive/nlp-project-clips/`):
1. `secrets/.env`: a copy of your Mac's `.env` (Groq, Hugging Face, Gemini keys). Colab Secrets only work in the Colab website, so the keys come from this private Drive file instead. Never put it anywhere else.
2. `testset/`: the clips (`hi_01.mp4`, `hi_02.mp4`, `ta_01.mp4`, `ta_02.mp4` from the Mac's `data/clips/`).

The project runs in its own Python 3.11 environment (`.venv`) with the versions pinned in step 3; Colab's own Python (3.13, transformers 5) is only used to drive the notebook.

In [23]:
# 1. Check the GPU
import subprocess, sys, platform
print("python", sys.version.split()[0], "|", platform.node())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU visible: Runtime -> change to T4 GPU")

python 3.13.15 | bd6351db38a8
Tesla T4, 15360 MiB, 580.82.07



In [24]:
# 2. Mount Google Drive and set paths
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/nlp-project-clips")
REPO = Path("/content/nlp-project")
# Hugging Face models are cached on Drive, so later sessions don't download them again.
os.environ["HF_HOME"] = str(DRIVE / "hf-cache")
for sub in ["secrets", "testset", "step2-tts", "hf-cache", "outputs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Drive folder:", sorted(p.name for p in DRIVE.iterdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folder: ['hf-cache', 'outputs', 'secrets', 'step2-tts', 'testset']


In [25]:
# 3. Get the latest code from GitHub (public repo)
if (REPO / ".git").exists():
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/thisisaadi123/nlp-project.git {REPO}
!git -C {REPO} log --oneline -3

remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 67 (delta 13), reused 67 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (67/67), 94.38 KiB | 785.00 KiB/s, done.
From https://github.com/thisisaadi123/nlp-project
   bdc4af2..ba8f687  main       -> origin/main
Updating bdc4af2..ba8f687
Fast-forward
 .gitignore                                    |    6 +
 README.md                                     |   13 +-
 backend/api/app.py                            |  188 ++
 frontend/.gitkeep => backend/game/__init__.py |    0
 backend/game/analyzer.py                      |   59 +
 backend/game/coach.py                         |   24 +
 backend/game/library.py                       |  255 +++
 backend/game/play.py                          |  113 +
 backend/game/premiere.py                      |   34 +
 backend/game/shape.py                         |   60 +
 backend/scoring/compare.py  

In [26]:
# 4. Keys: copy the private .env from Drive into the project (the values are never printed)
import shutil
env_src = DRIVE / "secrets" / ".env"
if not env_src.exists():
    raise FileNotFoundError(f"Put a copy of your .env at {env_src} (Drive: nlp-project-clips/secrets/.env)")
shutil.copy(env_src, REPO / ".env")
names = [line.split("=", 1)[0] for line in (REPO / ".env").read_text().splitlines()
         if "=" in line and not line.startswith("#") and line.split("=", 1)[1].strip()]
print("keys present:", names)

keys present: ['GROQ_API_KEY', 'HF_TOKEN', 'GEMINI_API_KEY']


In [27]:
%%bash
# 5. Python 3.11 environment with the pinned GPU stack (~3-5 min)
set -e
cd /content/nlp-project
pip install -q uv
uv python install 3.11 -q
[ -d .venv ] || uv venv -q --python 3.11 .venv
uv pip install -q --python .venv/bin/python -r requirements-gpu.txt --override overrides-gpu.txt
apt-get -qq install -y ffmpeg sox libsndfile1 > /dev/null
.venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

torch 2.14.1+cu130 | CUDA: True | Tesla T4


In [28]:
# 6. Copy the clips from Drive into the project
clips = REPO / "data" / "clips"
clips.mkdir(parents=True, exist_ok=True)
for f in (DRIVE / "testset").glob("*.mp4"):
    shutil.copy(f, clips / f.name)
tts = clips / "step2" / "tts"
tts.mkdir(parents=True, exist_ok=True)
for f in (DRIVE / "step2-tts").glob("*.wav"):
    shutil.copy(f, tts / f.name)
print("clips:", sorted(p.name for p in clips.glob("*.mp4")) or "none: upload them to Drive nlp-project-clips/testset/")
print("tts takes:", len(list(tts.glob("*.wav"))))

clips: ['hi_01.mp4', 'hi_02.mp4', 'ta_01.mp4', 'ta_02.mp4']
tts takes: 9


In [29]:
# 7. Check keys, model access and the GPU from inside the project environment
!cd {REPO} && HF_HOME={os.environ['HF_HOME']} .venv/bin/python scripts/check_setup.py 2>&1 | grep -v "Live Groq models"

[PASS] Python — 3.11.17
[PASS] ffmpeg — ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
[PASS] config.yaml — languages: Hindi, Bengali, Marathi, Telugu, Tamil
[PASS] Groq text_model — openai/gpt-oss-120b
[PASS] Groq vision_model — qwen/qwen3.8-27b
[PASS] Groq test call — அண்ணா
[PASS] Hugging Face — logged in as thisisaadi123
[PASS] HF access pyannote/speaker-diarization-3.1
[PASS] HF access pyannote/segmentation-3.0
[PASS] HF access pyannote/speaker-diarization-community-1
[PASS] HF access ai4bharat/indic-conformer-600m-multilingual
[PASS] HF access ai4bharat/indictrans2-indic-indic-dist-320M
[PASS] HF access ai4bharat/indic-parler-tts
[PASS] Gemini (backup) — 50 models visible
[PASS] GPU — Tesla T4, 14.6 GB VRAM, torch 2.14.1+cu130

15 passed, 0 warnings, 0 failed
